In [1]:

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
data_dir = Path("../data/processed")

products_df = pd.read_csv(
    data_dir / "product_info_cleaned.csv",
    dtype={"product_id": str}
).reset_index(drop=True)

train_df = pd.read_csv(
    data_dir / "train_interactions.csv",
    dtype={"author_id": str, "product_id": str}
)

test_df = pd.read_csv(
    data_dir / "test_interactions.csv",
    dtype={"author_id": str, "product_id": str}
)

print("FINAL MODEL EVALUATION")
print("-" * 40)

print("Products:", len(products_df))
print("Training interactions:", len(train_df))
print("Testing interactions:", len(test_df))
print("Training users:", train_df["author_id"].nunique())
print("Testing users:", test_df["author_id"].nunique())


FINAL MODEL EVALUATION
----------------------------------------
Products: 8494
Training interactions: 213982
Testing interactions: 66871
Training users: 30061
Testing users: 30061


In [2]:

# Reconstruct the previous development sample

common_users = pd.Index(
    train_df["author_id"].unique()
).intersection(
    test_df["author_id"].unique()
)

development_users = (
    pd.Series(common_users)
    .sample(
        n=min(100, len(common_users)),
        random_state=42
    )
    .tolist()
)

# Exclude development users
remaining_users = pd.Series(
    common_users[~common_users.isin(development_users)]
)

# Select an independent final evaluation sample
final_users = remaining_users.sample(
    n=min(500, len(remaining_users)),
    random_state=123
).tolist()

print("DATASET SPLIT CHECK")
print("-" * 40)

print("Development users:", len(development_users))
print("Final evaluation users:", len(final_users))

print(
    "User overlap:",
    len(set(development_users) & set(final_users))
)

assert not set(development_users) & set(final_users)

# Check interaction leakage
train_pairs = set(
    zip(train_df["author_id"], train_df["product_id"])
)

test_pairs = set(
    zip(test_df["author_id"], test_df["product_id"])
)

print("Train/Test interaction overlap:",
      len(train_pairs & test_pairs))

assert not train_pairs & test_pairs


DATASET SPLIT CHECK
----------------------------------------
Development users: 100
Final evaluation users: 500
User overlap: 0
Train/Test interaction overlap: 0


In [3]:

# Baseline TF-IDF Model

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    min_df=2,
    ngram_range=(1, 2),
    dtype=np.float32
)

tfidf_matrix = tfidf.fit_transform(
    products_df["combined_features"].fillna("")
)

product_indices = {
    pid: idx
    for idx, pid in enumerate(products_df["product_id"])
}

train_history = (
    train_df.groupby("author_id")["product_id"]
    .apply(list)
    .to_dict()
)

test_history = (
    test_df.groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("TF-IDF MATRIX:", tfidf_matrix.shape)
print("MODEL READY!")


TF-IDF MATRIX: (8494, 10000)
MODEL READY!


In [4]:

# Popularity-based recommendation ranking

popular_products = (
    train_df.groupby("product_id")["author_id"]
    .nunique()
    .sort_values(ascending=False)
    .index.tolist()
)

# Content-based recommendation
def recommend_content(user_id, k=5):

    liked_ids = train_history.get(str(user_id), [])

    liked_indices = [
        product_indices[pid]
        for pid in liked_ids
        if pid in product_indices
    ]

    if not liked_indices:
        return []

    profile = np.asarray(
        tfidf_matrix[liked_indices].mean(axis=0),
        dtype=np.float32
    )

    scores = cosine_similarity(
        profile,
        tfidf_matrix
    ).ravel()

    scores[liked_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]

    return products_df.iloc[
        top_indices
    ]["product_id"].tolist()


# Popularity-based recommendation
def recommend_popularity(user_id, k=5):

    seen_products = set(
        train_history.get(str(user_id), [])
    )

    recommendations = []

    for pid in popular_products:

        if pid not in seen_products:
            recommendations.append(pid)

        if len(recommendations) == k:
            break

    return recommendations


print("Recommendation functions ready!")


Recommendation functions ready!


In [5]:

# Evaluation metrics

def calculate_metrics(recommended, relevant, k=5):

    if not relevant:
        return None

    hits = [
        int(pid in relevant)
        for pid in recommended[:k]
    ]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant)

    cumulative_hits = 0
    precision_sum = 0.0

    for rank, hit in enumerate(hits, start=1):

        if hit:
            cumulative_hits += 1
            precision_sum += cumulative_hits / rank

    average_precision = (
        precision_sum / min(len(relevant), k)
    )

    return {
        "Precision@5": precision,
        "Recall@5": recall,
        "MAP@5": average_precision,
        "Hits": sum(hits)
    }


def evaluate_recommender(recommend_function, users):

    results = []

    for user_id in users:

        recommendations = recommend_function(
            user_id, k=5
        )

        relevant = test_history.get(
            str(user_id), set()
        )

        metrics = calculate_metrics(
            recommendations, relevant, k=5
        )

        if metrics is not None:
            results.append(metrics)

    return pd.DataFrame(results)


print("Evaluation functions ready!")


Evaluation functions ready!


In [6]:

# Final evaluation

print("Evaluating Content-Based TF-IDF...")

content_results = evaluate_recommender(
    recommend_content,
    final_users
)

print("Evaluating Popularity Baseline...")

popularity_results = evaluate_recommender(
    recommend_popularity,
    final_users
)

# Calculate average metrics
final_comparison = pd.DataFrame({
    "Content-Based TF-IDF": content_results.mean(),
    "Popularity Baseline": popularity_results.mean()
}).T

print("\nFINAL MODEL COMPARISON")
print("-" * 45)

print("Evaluated users:", len(content_results))

display(final_comparison.round(4))


Evaluating Content-Based TF-IDF...
Evaluating Popularity Baseline...

FINAL MODEL COMPARISON
---------------------------------------------
Evaluated users: 500


,Precision@5,Recall@5,MAP@5,Hits
Content-Based TF-IDF,0.0548,0.1752,0.1063,0.274
Popularity Baseline,0.0140,0.0300,0.0243,0.070


In [7]:

# Save final evaluation results

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_comparison.to_csv(
    output_dir / "final_model_comparison.csv"
)

print("Final evaluation saved successfully!")

print("\nFINAL RESULTS")
display(final_comparison.round(4))

print("\nBest model by MAP@5:")
print(final_comparison["MAP@5"].idxmax())


Final evaluation saved successfully!

FINAL RESULTS


,Precision@5,Recall@5,MAP@5,Hits
Content-Based TF-IDF,0.0548,0.1752,0.1063,0.274
Popularity Baseline,0.0140,0.0300,0.0243,0.070



Best model by MAP@5:
Content-Based TF-IDF
